# 사전 설정

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
CSV_PATH = "inbody_dataset.csv"

# 시스템에 설치된 폰트 중에서 한글 폰트 찾기
font_list = [f.name for f in fm.fontManager.ttflist]
korean_fonts = [f for f in font_list if 'Gothic' in f or 'Malgun' in f]

if korean_fonts:
    plt.rcParams['font.family'] = korean_fonts[0]
    print(f"사용 폰트: {korean_fonts[0]}")
else:
    print("한글 폰트를 찾을 수 없습니다.")

plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드

In [ ]:
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

In [ ]:
df['날짜'] = pd.to_datetime(df['날짜'], errors='coerce')

numeric_columns = [
    '키', '골격근량', '체수분', '단백질',
    '무기질', '체지방량', '체중',
    '복부지방률', '내장지방레벨'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

df = df.dropna(subset=['날짜'])
df = df.sort_values('날짜').reset_index(drop=True)

print(df.head())
print("\n=== 결측치 확인 ===")
print(df.isnull().sum())

# 특성 추가

In [ ]:
df['BMI'] = df['체중'] / ((df['키'] / 100) ** 2)
df['체지방률'] = df['체지방량'] / df['체중'] * 100
df['제지방량'] = df['체중'] - df['체지방량']

# 통계 분석

In [ ]:
# 최초 대비 최근 변화량
change_columns = [
    '체중', '골격근량', '체지방량',
    '체지방률', 'BMI', '복부지방률',
    '내장지방레벨'
]

print("최초 대비 최근 변화량")

for col in change_columns:
    first = df[col].iloc[0]
    last = df[col].iloc[-1]

    if pd.notna(first) and pd.notna(last):
        print(f"{col}: {first:.2f} -> {last:.2f} (변화: {last - first:+.2f})")

# 시각화

In [ ]:
import matplotlib
print(matplotlib.rcParams['font.family'])

In [ ]:
# 체중 및 골격근량 변화
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(df['날짜'], df['체중'],
             marker='o', label='체중')

axes[0].set_title('날짜별 체중 변화')
axes[0].set_ylabel('체중 (kg)')
axes[0].legend()

axes[1].plot(df['날짜'], df['골격근량'],
             marker='o', color='green', label='골격근량')

axes[1].set_title('날짜별 골격근량 변화')
axes[1].set_ylabel('골격근량 (kg)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(df['날짜'], df['체지방량'],
             marker='o', label='체지방량')

axes[0].set_title('날짜별 체지방량 변화')
axes[0].set_ylabel('체지방량 (kg)')
axes[0].legend()

axes[1].plot(df['날짜'], df['체지방률'],
             marker='o', color='red', label='체지방률')

axes[1].set_title('날짜별 체지방률 변화')
axes[1].set_ylabel('체지방률 (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(df['날짜'], df['복부지방률'],
             marker='o', label='복부지방률')

axes[0].set_title('날짜별 복부지방률 변화')
axes[0].legend()

axes[1].plot(df['날짜'], df['내장지방레벨'],
             marker='o', color='purple', label='내장지방레벨')

axes[1].set_title('날짜별 내장지방레벨 변화')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
correlation_columns = [
    '체중', '골격근량', '체수분', '단백질',
    '무기질', '체지방량', '체지방률',
    '복부지방률', '내장지방레벨', 'BMI'
]

corr = df[correlation_columns].corr(method='pearson')

print("\n=== 상관관계 행렬 ===")
print(corr.round(3))

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title('체성분 변수 간 상관관계')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.regplot(
    data=df,
    x='체중',
    y='골격근량',
    ax=axes[0]
)
axes[0].set_title('체중과 골격근량')

sns.regplot(
    data=df,
    x='체중',
    y='체지방량',
    ax=axes[1]
)
axes[1].set_title('체중과 체지방량')

sns.regplot(
    data=df,
    x='골격근량',
    y='체지방량',
    ax=axes[2]
)
axes[2].set_title('골격근량과 체지방량')

plt.tight_layout()
plt.show()